In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import ticker 
import pickle
import re
import pymongo
import sncosmo
from warptemplate import get_ztftable_from_ampel, get_warpedTimeSeriesModel, get_template_correction, WarpfitTemplateLoader
from scipy.stats.distributions import chi2

What happens here:
- Pick a BTS SN
- Load fp data from dp + anc info.
- Decide on whether to use a narrow or wide target type classification
- Decide on whether to allow templates based on the target SN to be used
- Decide on whether to fit using all available templates, or a random (or one specfic?)
- Make a nice figure!

In [ ]:
snname = 'ZTF20aayxdse'

In [ ]:
df_bts = pd.read_csv('/Users/jnordin/data/ztf/bts/bts_explorer_241122.csv')

In [ ]:
sndict = df_bts.loc[df_bts['ZTFID']==snname,:].iloc[0]

In [ ]:
print('looking at {} - type {}'.format(snname, sndict['type']))

In [ ]:
# NOw you have to decide which warp class to fit against:
warpclasses = [
    'SN Ib', 'SN Ia-91bg', 'SN II', 'SN Ia-91T',
    'SN Ib/c', 'SN Ia-pec', 'SN IIP', 'SN Iax', 'SN Ic',
    'SLSN-II']

In [ ]:
template_class_id = 2

In [ ]:
print('Fitting to templates of class', warpclasses[template_class_id])

In [ ]:
client = pymongo.MongoClient()
db = client.bts_ipacfp_strictbase    # Only final lc. try bts_ipacfp_strictbase_full for all alerts

In [ ]:
# Lets grab some data, starting with the target SN
tab = get_ztftable_from_ampel( snname, db, 
                              redshift=float(sndict['redshift']), 
                              type = sndict['type'] )

In [ ]:
# Parameters for fit template retrieval
exclude_input = [snname] # Will reject any warptemplate containing any of these (either as sn or template basis)
# How to define templates?
# - How many templates per sn basis? 
#      * if 'all' it will return one copy of each template, 
#.     * if int it will return that many, drawn according to the template probability, 
#      * if -int it will return that many copies drawn from a uniform probabilitiy
#      Note: draws made with replacement, so multiple copies can be returned if int is larger than the available number of templates (often 3)
template_selection = 'all'
# - How many sn basis?
#.     * if 'all', take one of each
#.     * if an int, draw these randomly (with replacement)
#.     Note: how many templates are returned is decided by the above parameter.
snbasis_selection = 'all'

warpdir = '/Users/jnordin/data/models/sncosmo/warpmod'

In [ ]:
warploader = WarpfitTemplateLoader(warpdir)

In [ ]:
templates2 = warploader.get_templates(
    fitclass=warpclasses[template_class_id],
    exclude_input = exclude_input, 
    template_selection=template_selection,
    snbasis_selection=snbasis_selection,
    random_seed=42
)

In [ ]:
# Loop through templates, fit and plot the best...
bestmodel = None
for k, template in enumerate(templates2):
    print(k,len(templates2),template)
    fitprop = ['t0', 'amplitude', 'hostebv']
    try:
        wresult, wfitted_model = sncosmo.fit_lc(
                        tab, template['model'],
                        fitprop,  # parameters of model to vary
                    )
    except RuntimeError:
        print('fit failed')
        continue
    # Evaluate
    if wresult['success']:
        if bestmodel is None:
            bestmodel = [template,wfitted_model, wresult['chisq'] / wresult['ndof']]
        elif wresult['chisq'] / wresult['ndof'] < bestmodel[2]:
            bestmodel = [template,wfitted_model, wresult['chisq'] / wresult['ndof']]
#    sncosmo.plot_lc(tab, wfitted_model)

So, fitting will be very slow but that is really not the main use of this. Can do it, but with a lot of iteration. Much easier to use to simulate lots of transients.

In [ ]:
bestmodel

In [ ]:
sncosmo.plot_lc(tab, bestmodel[1])

In [ ]:
bestmodel

In [ ]:
_ = sncosmo.plot_lc(tab, bestmodel[1])

In [ ]:
def get_warpfit_templates( 
    str: fitclass, 
    list: exclude_input = [],
    int | str: template_selection = 1,
    int | str: snbasis_selection = 1,
    bool: require_good_templatefit = False,
    str: warpcoeffs_dir = '/Users/jnordin/data/models/sncosmo/warpmod' 
        ) -> list[dict]:
    """
    # Parameters for fit template retrieval
    exclude_input = [snname] # Will reject any warptemplate containing any of these (either as sn or template basis)
   - How many templates per sn basis? 
      * if 'all' it will return one copy of each template, 
      * if int it will return that many, drawn according to the template probability, 
      * if -int it will return that many copies drawn from a uniform probabilitiy
      Note: draws made with replacement, so multiple copies can be returned if int is larger than the available number of templates (often 3)
    - How many sn basis?
         * if 'all', take one of each
         * if an int, draw these randomly (with replacement)
         Note: how many templates are returned is decided by the above parameter.
    require_bood_templatefit (bool): require that all included warpfits have P(chi)>0.01
     warpcoeffs_dir: directory where warpcoeff files are stored
     """

    # Load coeffs
    # Should have some checks here, possibly have this as class and loaded in memory if this is repeated for many transients
    storefile = os.path.join(warpcoeffs_dir, 'warpcoeffs_'+re.sub(r'/', '', warpclasses[classnbr])+'.pkl')
    with open(storefile, 'wb') as file:
        template_collection = pickle.load(file)

    # Start looping through potential sn basis
    for sndict in template_collection:
        if sndict['ztfid'] in exclude_input:
            print('Templates based on this SN - skipping')
            continue

        if require_good_templatefit and not sndict['good_fit']:
            print('Templates not all good fit, as required. Skipping.')
            continue

        # Loop through templates belonging to this sn
        possible_templates = []
        for warpfit in sndict['warpmodels']:
            # Check whether this template was excluded 
            if sndict['ztfid']  in exclude_input:
                print('excluded template - skipping')
                continue

            # Lookf like we wish to use this template
            possible_templates.append(
                {
                    'basis_sn': sndict['ztfid'], 
                    'model':  get_warpedTimeSeriesModel(
                                name=sndict['ztfid'] +'_'+sndict['ztfid'] , 
                                original_template_name=sndict['ztfid'], 
                                warpdata= warpfit['mdict'],
                                z=float(warpfit['z']),
                                original_template_version=None
                            ),
                    'tample_prob': warpfit['draw_prob']
                }
            )

        return possible_templates

    

In [ ]:
import os
import re
import pickle
import random
from typing import List, Dict, Union


def get_warpfit_templates(
    fitclass: str,
    exclude_input: list | None = None,
    template_selection: Union[int, str] = 1,
    snbasis_selection: Union[int, str] = 1,
    require_good_templatefit: bool = False,
    warpcoeffs_dir: str = '/Users/jnordin/data/models/sncosmo/warpmod'
) -> List[Dict]:
    """
    Retrieve warped template models.

    Parameters
    ----------
    exclude_input : list
        SN names to exclude (both as basis and templates)
    template_selection : int | 'all'
        Number of templates per SN basis
    snbasis_selection : int | 'all'
        Number of SN bases to draw
    require_good_templatefit : bool
        Require P(chi) > 0.01
    """

    exclude_input = exclude_input or []

    # Load coefficients
    storefile = os.path.join(
        warpcoeffs_dir,
        f'warpcoeffs_{re.sub(r"/", "", fitclass)}.pkl'
    )

    with open(storefile, 'rb') as file:
        template_collection = pickle.load(file)

    # --- Filter SN bases ---
    valid_snbases = []
    for sndict in template_collection:
        if sndict['ztfid'] in exclude_input:
            continue

        if require_good_templatefit and not sndict.get('good_fit', False):
            continue

        valid_snbases.append(sndict)

    if not valid_snbases:
        return []

    # --- Select SN bases ---
    if snbasis_selection == 'all':
        selected_snbases = valid_snbases
    elif isinstance(snbasis_selection, int):
        selected_snbases = random.choices(valid_snbases, k=snbasis_selection)
    else:
        raise ValueError("snbasis_selection must be int or 'all'")

    results = []

    # --- Loop over selected SN bases ---
    for sndict in selected_snbases:

        possible_templates = []

        for warpfit in sndict['warpmodels']:

            # (Optional) exclude templates if needed (if template ID exists)
            # Example assumes warpfit might contain template name
            if warpfit.get('model') in exclude_input:
                continue

            possible_templates.append({
                'basis_sn': sndict['ztfid'],
                'model': get_warpedTimeSeriesModel(
                    name=f"{sndict['ztfid']}_{warpfit.get('model', 'tpl')}",
                    original_template_name=warpfit.get('model'),
                    warpdata=warpfit['mdict'],
                    z=float(warpfit['z']),
                    original_template_version=None
                ),
                'template_prob': warpfit['draw_prob']
            })

        if not possible_templates:
            continue

        # --- Template selection logic ---
        if template_selection == 'all':
            selected_templates = possible_templates

        elif isinstance(template_selection, int):

            if template_selection > 0:
                weights = [tpl['template_prob'] for tpl in possible_templates]
                selected_templates = random.choices(
                    possible_templates,
                    weights=weights,
                    k=template_selection
                )
            else:
                # uniform selection
                selected_templates = random.choices(
                    possible_templates,
                    k=abs(template_selection)
                )

        else:
            raise ValueError("template_selection must be int or 'all'")

        results.extend(selected_templates)

    return results

In [ ]:
templates = get_warpfit_templates('SN II', exclude_input = ['ZTF20aayxdse'], 
                                  template_selection=1,snbasis_selection='all'
                                 )

In [ ]:
len(templates)

In [ ]:
templates[0]

In [ ]:
import os
import re
import pickle
import random
import logging
from typing import List, Dict, Union, Optional


class WarpfitTemplateLoader:
    def __init__(
        self,
        warpcoeffs_dir: str,
        logger: Optional[logging.Logger] = None
    ):
        self.warpcoeffs_dir = warpcoeffs_dir
        self._cache: Dict[str, list] = {}

        if logger is None:
            logging.basicConfig(
                level=logging.INFO,
                format="%(asctime)s [%(levelname)s] %(message)s"
            )
            self.logger = logging.getLogger(__name__)
        else:
            self.logger = logger

    # -------------------------
    # Internal: load with cache
    # -------------------------
    def _load_coeffs(self, fitclass: str) -> list:
        key = re.sub(r"/", "", fitclass)

        if key in self._cache:
            self.logger.debug(f"Cache hit for fitclass={key}")
            return self._cache[key]

        filepath = os.path.join(
            self.warpcoeffs_dir,
            f"warpcoeffs_{key}.pkl"
        )

        self.logger.info(f"Loading warpcoeffs from {filepath}")

        if not os.path.exists(filepath):
            self.logger.error(f"File not found: {filepath}")
            raise FileNotFoundError(filepath)

        with open(filepath, "rb") as f:
            data = pickle.load(f)

        self._cache[key] = data
        return data

    def get_templates(
        self,
        fitclass: str,
        exclude_input: Optional[list] = None,
        template_selection: Union[int, str] = 1,
        snbasis_selection: Union[int, str] = 1,
        require_good_templatefit: bool = False,
        random_seed: Optional[int] = None,   # <-- NEW
    ) -> List[Dict]:
    
        exclude_input = exclude_input or []
    
        # -------------------------
        # Local RNG (reproducible)
        # -------------------------
        rng = random.Random(random_seed)
    
        if random_seed is not None:
            self.logger.info(f"Using random seed: {random_seed}")
    
        template_collection = self._load_coeffs(fitclass)
    
        # -------------------------
        # Filter SN bases
        # -------------------------
        valid_snbases = []
        for sndict in template_collection:
    
            sn_name = sndict.get("ztfid")
    
            if sn_name in exclude_input:
                self.logger.debug(f"Excluded SN basis: {sn_name}")
                continue
    
            if require_good_templatefit and not sndict.get("good_fit", False):
                self.logger.debug(f"Rejected (bad fit): {sn_name}")
                continue
    
            valid_snbases.append(sndict)
    
        if not valid_snbases:
            self.logger.warning("No valid SN bases after filtering")
            return []
    
        self.logger.info(f"{len(valid_snbases)} SN bases available after filtering")
    
        # -------------------------
        # Select SN bases
        # -------------------------
        if snbasis_selection == "all":
            selected_snbases = valid_snbases
    
        elif isinstance(snbasis_selection, int):
            selected_snbases = rng.choices(
                valid_snbases,
                k=snbasis_selection
            )
    
        else:
            raise ValueError("snbasis_selection must be int or 'all'")
    
        results = []
    
        # -------------------------
        # Loop SN bases
        # -------------------------
        for sndict in selected_snbases:
            sn_name = sndict["ztfid"]
    
            possible_templates = []
    
            for warpfit in sndict["warpmodels"]:
    
                template_sn = warpfit.get("model")
    
                if template_sn in exclude_input:
                    self.logger.debug(
                        f"Excluded template {template_sn} (basis {sn_name})"
                    )
                    continue
    
                try:
                    model = get_warpedTimeSeriesModel(
                        name=f"{sn_name}_{template_sn or 'tpl'}",
                        original_template_name=template_sn,
                        warpdata=warpfit["mdict"],
                        z=float(warpfit["z"]),
                        original_template_version=None,
                    )
                except Exception as e:
                    self.logger.error(
                        f"Model construction failed for {sn_name}: {e}"
                    )
                    continue
    
                possible_templates.append({
                    "basis_sn": sn_name,
                    "model": model,
                    "template_prob": warpfit["draw_prob"],
                })
    
            if not possible_templates:
                self.logger.debug(f"No valid templates for SN {sn_name}")
                continue
    
            # -------------------------
            # Template selection
            # -------------------------
            if template_selection == "all":
                selected_templates = possible_templates
    
            elif isinstance(template_selection, int):
    
                if template_selection > 0:
                    weights = [tpl["template_prob"] for tpl in possible_templates]
    
                    selected_templates = rng.choices(
                        possible_templates,
                        weights=weights,
                        k=template_selection
                    )
    
                else:
                    selected_templates = rng.choices(
                        possible_templates,
                        k=abs(template_selection)
                    )
    
            else:
                raise ValueError("template_selection must be int or 'all'")
    
            results.extend(selected_templates)
    
        self.logger.info(f"Returning {len(results)} templates")
    
        return results

    # -------------------------
    # Optional: cache control
    # -------------------------
    def clear_cache(self):
        self.logger.info("Clearing warpcoeff cache")
        self._cache.clear()

In [ ]:
warploader = WarpfitTemplateLoader('/Users/jnordin/data/models/sncosmo/warpmod')

In [ ]:
templates2 = warploader.get_templates(
    fitclass="SN II",
    exclude_input = ['ZTF20aayxdse'], 
    template_selection=1,
    snbasis_selection=1,
    random_seed=42
)


In [ ]:
warploader.